<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/Graph_representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ts2vg
!pip install mne
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from ts2vg import HorizontalVG
import scipy.sparse as sp

In [ ]:
def build_graph_representations(edge_list, node_index, node_labels, graph_meta):
    n_nodes_total = graph_meta["n_nodes_total"]
    time_values   = graph_meta["time_index"]

    # 1. SPARSE ADJACENCY
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    for _, row in edge_list.iterrows():
        u, v = int(row["source"]), int(row["target"])
        adj_sparse[u, v] = 1
        adj_sparse[v, u] = 1  # undirected

    # 2. DENSE ADJACENCY DATAFRAME
    adj_matrix = pd.DataFrame(
        adj_sparse.toarray(),
        index=node_labels,
        columns=node_labels
    )

    # 3. NODE TABLE
    node_rows = []
    for (electrode, t), node_id in node_index.items():
        node_rows.append({
            "node_id": node_id,
            "electrode": electrode,
            "time_idx": t,
            "time_s": float(time_values[t]),
            "node_label": f"{electrode}_t{t}",
        })

    node_table = pd.DataFrame(node_rows).sort_values("node_id").reset_index(drop=True)

    return adj_sparse, adj_matrix, node_table